In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
from faker import Faker
import random

/home/user/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


### AI Generated / Engineered Saas Data

In [2]:
# Set seed for reproducibility
np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

print("🚀 Starting SaaS Data Generation...")
print("=" * 50)

# Configuration
NUM_ACCOUNTS = 2000
START_DATE = datetime(2022, 1, 1)
END_DATE = datetime(2024, 12, 31)

# Industry distribution
INDUSTRIES = ['SaaS', 'E-commerce', 'Finance', 'Healthcare', 'Education', 
              'Marketing', 'Real Estate', 'Manufacturing', 'Consulting', 'Media']

# Plan tiers with realistic pricing
PLAN_TIERS = {
    'Basic': {'min': 49, 'max': 299, 'avg': 150},
    'Pro': {'min': 299, 'max': 1999, 'avg': 800},
    'Enterprise': {'min': 2000, 'max': 15000, 'avg': 5000}
}

# Acquisition channels
CHANNELS = ['Organic Search', 'Paid Search', 'LinkedIn', 'Content Marketing', 
            'Partnership', 'Referral', 'Direct', 'Events']

# Features
FEATURES = [
    'Dashboard Export', 'API Access', 'Team Collaboration', 'Custom Reports',
    'Integrations', 'Advanced Analytics', 'Mobile App', 'SSO',
    'White Labeling', 'Webhook', 'Bulk Import', 'Auto Backup',
    'Custom Fields', 'Role Permissions', 'Email Automation'
]

# Churn reasons
CHURN_REASONS = {
    'Price Too High': 0.30,
    'Missing Features': 0.25,
    'Switched to Competitor': 0.20,
    'No Longer Needed': 0.15,
    'Poor Customer Support': 0.10
}

print("✓ Configuration loaded")

# ============================================
# 1. GENERATE ACCOUNTS
# ============================================
print("\n📊 Generating Accounts...")

accounts_data = []

for i in range(1, NUM_ACCOUNTS + 1):
    # Signup date (more recent = more accounts, realistic growth)
    days_from_start = int(np.random.exponential(scale=400))
    signup_date = START_DATE + timedelta(days=min(days_from_start, 
                                                   (END_DATE - START_DATE).days))
    
    # Plan tier (40% Basic, 35% Pro, 25% Enterprise)
    plan_tier = np.random.choice(['Basic', 'Pro', 'Enterprise'], 
                                 p=[0.40, 0.35, 0.25])
    
    # Company size correlates with plan tier
    if plan_tier == 'Basic':
        company_size = np.random.choice(['1-10', '11-50'], p=[0.7, 0.3])
        seats = np.random.randint(1, 15)
    elif plan_tier == 'Pro':
        company_size = np.random.choice(['11-50', '51-200'], p=[0.6, 0.4])
        seats = np.random.randint(5, 80)
    else:  # Enterprise
        company_size = np.random.choice(['51-200', '201-1000', '1000+'], p=[0.5, 0.3, 0.2])
        seats = np.random.randint(20, 500)
    
    account = {
        'account_id': f'ACC_{i:05d}',
        'account_name': fake.company(),
        'industry': np.random.choice(INDUSTRIES),
        'country': np.random.choice(['United States', 'United Kingdom', 'Canada', 
                                     'Australia', 'Germany'], p=[0.60, 0.20, 0.10, 0.05, 0.05]),
        'signup_date': signup_date.strftime('%Y-%m-%d'),
        'referral_source': np.random.choice(CHANNELS, p=[0.25, 0.20, 0.15, 0.12, 0.10, 0.08, 0.06, 0.04]),
        'plan_tier': plan_tier,
        'company_size': company_size,
        'seats': seats,
        'is_trial': 0  # All converted (we're past trial)
    }
    
    accounts_data.append(account)

accounts_df = pd.DataFrame(accounts_data)
print(f"✓ Generated {len(accounts_df)} accounts")

# ============================================
# 2. GENERATE SUBSCRIPTIONS + MRR MOVEMENTS
# ============================================
print("\n💰 Generating Subscription Events...")

subscriptions_data = []
mrr_movements_data = []
churn_events_data = []

churned_accounts = set()
sub_id = 1
movement_id = 1
churn_id = 1

for idx, account in accounts_df.iterrows():
    account_id = account['account_id']
    signup_date = pd.to_datetime(account['signup_date'])
    plan_tier = account['plan_tier']
    
    # Initial MRR
    tier_config = PLAN_TIERS[plan_tier]
    base_mrr = np.random.randint(tier_config['min'], tier_config['max'])
    seat_multiplier = 1 + (account['seats'] / 100) * 0.2
    current_mrr = int(base_mrr * seat_multiplier)
    current_plan = plan_tier
    
    # ====================================
    # 🆕 NEW MRR MOVEMENT - Customer signup
    # ====================================
    mrr_movements_data.append({
        'movement_id': f'MOV_{movement_id:05d}',
        'account_id': account_id,
        'movement_date': signup_date.strftime('%Y-%m-%d'),
        'movement_type': 'New',
        'mrr_before': 0,
        'mrr_after': current_mrr,
        'mrr_change': current_mrr,
        'plan_tier_before': None,
        'plan_tier_after': current_plan
    })
    movement_id += 1
    
    # Determine if account will churn (targeting 12% total churn rate)
    will_churn = np.random.random() < 0.12
    
    if will_churn:
        # Churn between 3-24 months after signup
        months_until_churn = np.random.randint(3, 25)
        churn_date = signup_date + timedelta(days=months_until_churn * 30)
        
        # Don't churn after END_DATE
        if churn_date > END_DATE:
            will_churn = False
            churn_date = None
    else:
        churn_date = None
    
    # Generate monthly subscription records
    current_date = signup_date
    last_upgrade_date = None
    has_expanded = False
    
    while current_date <= END_DATE:
        # Check if churned
        if will_churn and churn_date and current_date >= churn_date:
            # Create churn event
            reason = np.random.choice(list(CHURN_REASONS.keys()), 
                                     p=list(CHURN_REASONS.values()))
            
            # Possible downgrade before churn (30% chance)
            preceding_downgrade = 1 if np.random.random() < 0.30 else 0
            
            churn_events_data.append({
                'churn_id': f'CHN_{churn_id:05d}',
                'account_id': account_id,
                'subscription_id': f'SUB_{sub_id:05d}',
                'churn_date': churn_date.strftime('%Y-%m-%d'),
                'reason_code': reason,
                'mrr_lost': current_mrr,
                'preceding_upgrade_flag': 0,
                'preceding_downgrade_flag': preceding_downgrade,
                'is_reactivation': 0,
                'feedback_text': fake.sentence() if np.random.random() > 0.15 else None
            })
            
            churn_id += 1
            churned_accounts.add(account_id)
            
            # MRR Movement for churn
            mrr_movements_data.append({
                'movement_id': f'MOV_{movement_id:05d}',
                'account_id': account_id,
                'movement_date': churn_date.strftime('%Y-%m-%d'),
                'movement_type': 'Churn',
                'mrr_before': current_mrr,
                'mrr_after': 0,
                'mrr_change': -current_mrr,
                'plan_tier_before': current_plan,
                'plan_tier_after': None
            })
            movement_id += 1
            
            break  # Stop generating subscriptions
        
        # Active subscription record
        subscriptions_data.append({
            'subscription_id': f'SUB_{sub_id:05d}',
            'account_id': account_id,
            'plan_tier': current_plan,
            'mrr_amount': current_mrr,
            'arr_amount': current_mrr * 12,
            'start_date': current_date.strftime('%Y-%m-%d'),
            'end_date': None if not will_churn else (churn_date.strftime('%Y-%m-%d') 
                                                      if churn_date and current_date >= churn_date else None),
            'billing_frequency': np.random.choice(['Monthly', 'Annual'], p=[0.65, 0.35]),
            'auto_renew_flag': 1 if np.random.random() > 0.05 else 0
        })
        
        sub_id += 1
        
        # Expansion opportunity (10% chance per year for non-Enterprise)
        months_active = (current_date - signup_date).days / 30
        
        if (current_plan != 'Enterprise' and 
            months_active >= 6 and 
            not has_expanded and
            np.random.random() < 0.008):  # ~10% annual chance
            
            # Upgrade
            old_plan = current_plan
            old_mrr = current_mrr
            
            if current_plan == 'Basic':
                current_plan = 'Pro'
                new_tier_config = PLAN_TIERS['Pro']
            else:  # Pro to Enterprise
                current_plan = 'Enterprise'
                new_tier_config = PLAN_TIERS['Enterprise']
            
            current_mrr = int(np.random.randint(new_tier_config['min'], 
                                                new_tier_config['max']) * seat_multiplier)
            
            has_expanded = True
            last_upgrade_date = current_date
            
            # MRR Movement for expansion
            mrr_movements_data.append({
                'movement_id': f'MOV_{movement_id:05d}',
                'account_id': account_id,
                'movement_date': current_date.strftime('%Y-%m-%d'),
                'movement_type': 'Expansion',
                'mrr_before': old_mrr,
                'mrr_after': current_mrr,
                'mrr_change': current_mrr - old_mrr,
                'plan_tier_before': old_plan,
                'plan_tier_after': current_plan
            })
            movement_id += 1
        
        # Contraction (rare, 2% chance if approaching churn)
        elif (will_churn and churn_date and 
              (churn_date - current_date).days < 90 and 
              (churn_date - current_date).days > 30 and
              np.random.random() < 0.20):
            
            old_mrr = current_mrr
            current_mrr = int(current_mrr * 0.7)  # 30% reduction
            
            mrr_movements_data.append({
                'movement_id': f'MOV_{movement_id:05d}',
                'account_id': account_id,
                'movement_date': current_date.strftime('%Y-%m-%d'),
                'movement_type': 'Contraction',
                'mrr_before': old_mrr,
                'mrr_after': current_mrr,
                'mrr_change': current_mrr - old_mrr,
                'plan_tier_before': current_plan,
                'plan_tier_after': current_plan
            })
            movement_id += 1
        
        # Move to next month
        current_date = current_date + timedelta(days=30)

# Add churn flag to accounts
accounts_df['churn_flag'] = accounts_df['account_id'].isin(churned_accounts).astype(int)

subscriptions_df = pd.DataFrame(subscriptions_data)
mrr_movements_df = pd.DataFrame(mrr_movements_data)
churn_events_df = pd.DataFrame(churn_events_data)

print(f"✓ Generated {len(subscriptions_df)} subscription records")
print(f"✓ Generated {len(mrr_movements_df)} MRR movement events")
print(f"✓ Generated {len(churn_events_df)} churn events")

# ============================================
# 3. GENERATE FEATURE USAGE
# ============================================
print("\n🖱️ Generating Feature Usage...")

feature_usage_data = []
usage_id = 1

for idx, account in accounts_df.iterrows():
    account_id = account['account_id']
    signup_date = pd.to_datetime(account['signup_date'])
    is_churned = account['churn_flag'] == 1
    
    # Get churn date if applicable
    if is_churned:
        churn_info = churn_events_df[churn_events_df['account_id'] == account_id]
        if not churn_info.empty:
            churn_date = pd.to_datetime(churn_info.iloc[0]['churn_date'])
        else:
            churn_date = END_DATE
    else:
        churn_date = END_DATE
    
    # Determine user engagement level
    engagement_level = np.random.choice(['low', 'medium', 'high'], p=[0.25, 0.50, 0.25])
    
    if engagement_level == 'low':
        features_used = np.random.choice(FEATURES, size=np.random.randint(2, 5), replace=False)
        usage_frequency = 0.3
    elif engagement_level == 'medium':
        features_used = np.random.choice(FEATURES, size=np.random.randint(4, 8), replace=False)
        usage_frequency = 0.6
    else:  # high
        features_used = np.random.choice(FEATURES, size=np.random.randint(7, len(FEATURES)), replace=False)
        usage_frequency = 0.9
    
    # Generate usage events
    current_date = signup_date
    
    while current_date <= min(churn_date, END_DATE):
        if np.random.random() < usage_frequency:
            for feature in features_used:
                # Declining usage before churn (ghosting pattern)
                if is_churned and (churn_date - current_date).days < 60:
                    days_to_churn = (churn_date - current_date).days
                    decline_factor = days_to_churn / 60
                    if np.random.random() > decline_factor:
                        continue
                
                usage_count = np.random.randint(1, 20) if np.random.random() > 0.3 else np.random.randint(1, 5)
                usage_duration = usage_count * np.random.randint(30, 600)
                error_count = 1 if np.random.random() < 0.05 else 0
                is_beta = 1 if 'Advanced' in feature or 'Custom' in feature else 0
                
                feature_usage_data.append({
                    'usage_id': f'USG_{usage_id:06d}',
                    'account_id': account_id,
                    'feature_name': feature,
                    'usage_date': current_date.strftime('%Y-%m-%d'),
                    'usage_count': usage_count,
                    'usage_duration_secs': usage_duration,
                    'error_count': error_count,
                    'is_beta_feature': is_beta
                })
                
                usage_id += 1
        
        current_date = current_date + timedelta(days=np.random.randint(1, 5))

feature_usage_df = pd.DataFrame(feature_usage_data)
print(f"✓ Generated {len(feature_usage_df)} feature usage events")

# ============================================
# 4. GENERATE SUPPORT TICKETS
# ============================================
print("\n🎫 Generating Support Tickets...")

support_tickets_data = []
ticket_id = 1

TICKET_CATEGORIES = ['Bug', 'Feature Request', 'How-To', 'Billing', 'Performance', 'Integration']
PRIORITIES = {'Low': 0.60, 'Medium': 0.25, 'High': 0.12, 'Critical': 0.03}

for idx, account in accounts_df.iterrows():
    account_id = account['account_id']
    signup_date = pd.to_datetime(account['signup_date'])
    is_churned = account['churn_flag'] == 1
    
    if is_churned:
        avg_tickets_per_year = np.random.randint(6, 15)
    else:
        avg_tickets_per_year = np.random.randint(2, 8)
    
    if is_churned:
        churn_info = churn_events_df[churn_events_df['account_id'] == account_id]
        if not churn_info.empty:
            end_date = pd.to_datetime(churn_info.iloc[0]['churn_date'])
        else:
            end_date = END_DATE
    else:
        end_date = END_DATE
    
    months_active = max(1, (end_date - signup_date).days / 30)
    total_tickets = int((months_active / 12) * avg_tickets_per_year)
    
    for _ in range(total_tickets):
        days_offset = np.random.randint(0, int((end_date - signup_date).days))
        created_date = signup_date + timedelta(days=days_offset)
        
        priority = np.random.choice(list(PRIORITIES.keys()), p=list(PRIORITIES.values()))
        category = np.random.choice(TICKET_CATEGORIES)
        
        if priority == 'Critical':
            resolution_hours = np.random.randint(1, 8)
        elif priority == 'High':
            resolution_hours = np.random.randint(4, 24)
        elif priority == 'Medium':
            resolution_hours = np.random.randint(12, 72)
        else:
            resolution_hours = np.random.randint(24, 168)
        
        resolved_date = created_date + timedelta(hours=resolution_hours)
        
        if np.random.random() < 0.80:
            if is_churned:
                satisfaction = np.random.choice([1, 2, 3, 4, 5], p=[0.15, 0.25, 0.35, 0.15, 0.10])
            else:
                satisfaction = np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.10, 0.20, 0.35, 0.30])
        else:
            satisfaction = None
        
        escalation = 1 if (is_churned and np.random.random() < 0.15) or (not is_churned and np.random.random() < 0.05) else 0
        
        support_tickets_data.append({
            'ticket_id': f'TKT_{ticket_id:06d}',
            'account_id': account_id,
            'created_date': created_date.strftime('%Y-%m-%d'),
            'resolved_date': resolved_date.strftime('%Y-%m-%d') if resolved_date <= END_DATE else None,
            'priority': priority,
            'category': category,
            'resolution_time_hours': resolution_hours if resolved_date <= END_DATE else None,
            'satisfaction_score': satisfaction,
            'escalation_flag': escalation
        })
        
        ticket_id += 1

support_tickets_df = pd.DataFrame(support_tickets_data)
print(f"✓ Generated {len(support_tickets_df)} support tickets")

# ============================================
# 5. SAVE TO CSV FILES
# ============================================
print("\n💾 Saving data to CSV files...")

accounts_df.to_csv('accounts.csv', index=False)
subscriptions_df.to_csv('subscriptions.csv', index=False)
mrr_movements_df.to_csv('mrr_movements.csv', index=False)
churn_events_df.to_csv('churn_events.csv', index=False)
feature_usage_df.to_csv('feature_usage.csv', index=False)
support_tickets_df.to_csv('support_tickets.csv', index=False)

print("\n" + "=" * 50)
print("✅ DATA GENERATION COMPLETE!")
print("=" * 50)
print(f"\n📊 Summary:")
print(f"   • Accounts: {len(accounts_df):,}")
print(f"   • Subscriptions: {len(subscriptions_df):,}")
print(f"   • MRR Movements: {len(mrr_movements_df):,}")
print(f"      - New: {len(mrr_movements_df[mrr_movements_df['movement_type']=='New']):,}")
print(f"      - Expansion: {len(mrr_movements_df[mrr_movements_df['movement_type']=='Expansion']):,}")
print(f"      - Contraction: {len(mrr_movements_df[mrr_movements_df['movement_type']=='Contraction']):,}")
print(f"      - Churn: {len(mrr_movements_df[mrr_movements_df['movement_type']=='Churn']):,}")
print(f"   • Churn Events: {len(churn_events_df):,}")
print(f"   • Feature Usage: {len(feature_usage_df):,}")
print(f"   • Support Tickets: {len(support_tickets_df):,}")
print(f"\n📁 Files saved in current directory")
print(f"\n🎯 Churn Rate: {(len(churn_events_df) / len(accounts_df) * 100):.1f}%")
print(f"🎯 Active Accounts: {len(accounts_df) - len(churn_events_df):,}")
print("\n✨ Ready to import into MySQL!")

🚀 Starting SaaS Data Generation...
✓ Configuration loaded

📊 Generating Accounts...
✓ Generated 2000 accounts

💰 Generating Subscription Events...
✓ Generated 46205 subscription records
✓ Generated 2456 MRR movement events
✓ Generated 214 churn events

🖱️ Generating Feature Usage...
✓ Generated 2351716 feature usage events

🎫 Generating Support Tickets...
✓ Generated 17268 support tickets

💾 Saving data to CSV files...

✅ DATA GENERATION COMPLETE!

📊 Summary:
   • Accounts: 2,000
   • Subscriptions: 46,205
   • MRR Movements: 2,456
      - New: 2,000
      - Expansion: 201
      - Contraction: 41
      - Churn: 214
   • Churn Events: 214
   • Feature Usage: 2,351,716
   • Support Tickets: 17,268

📁 Files saved in current directory

🎯 Churn Rate: 10.7%
🎯 Active Accounts: 1,786

✨ Ready to import into MySQL!


# 📊 **Velocity SaaS Dataset - Table Documentation**

# Velocity SaaS - Churn Analysis & Revenue Recovery

**Dataset:** Synthetically generated B2B SaaS data modeled after industry benchmarks

**Company Profile:**  
Velocity is a fictional B2B SaaS company providing analytics software to mid-market enterprises. This dataset represents 3 years of operational data (2022-2024) including 2,000 customer accounts, subscription movements, feature usage, and support interactions.

**Why Synthetic Data?**
I generated this dataset to demonstrate realistic SaaS analysis patterns while maintaining full control over data structure and business logic. All patterns are validated against industry benchmarks from OpenView, ChartMogul, and ProfitWell research.


---

## 🏛️ **1. Accounts Table**

| Column | Description |
|--------|-------------|
| **account_id** | Unique company identifier (e.g., `ACC_00001`) |
| **account_name** | Company name |
| **industry** | Business sector (SaaS, E-commerce, Finance, Healthcare, etc.) |
| **country** | Company location (US, UK, Canada, Australia, Germany) |
| **signup_date** | Date when account was created |
| **referral_source** | Acquisition channel (Organic Search, Paid Search, LinkedIn, Partnership, etc.) |
| **plan_tier** | Subscription level: Basic, Pro, or Enterprise |
| **company_size** | Employee count bracket (1-10, 11-50, 51-200, 201-1000, 1000+) |
| **seats** | Number of user licenses purchased |
| **is_trial** | Trial status: 0 = converted, 1 = trial (all accounts are converted) |
| **churn_flag** | Churn status: 0 = active, 1 = churned |

**Row Count:** 2,000 accounts

---

## 💳 **2. Subscriptions Table**

| Column | Description |
|--------|-------------|
| **subscription_id** | Unique subscription record ID (e.g., `SUB_00001`) |
| **account_id** | Links to accounts table |
| **plan_tier** | Current subscription tier (Basic, Pro, Enterprise) |
| **mrr_amount** | Monthly Recurring Revenue in dollars |
| **arr_amount** | Annual Recurring Revenue (MRR × 12) |
| **start_date** | When this subscription period started |
| **end_date** | When subscription ended (NULL = still active) |
| **billing_frequency** | Monthly or Annual payment cycle |
| **auto_renew_flag** | Auto-renewal enabled: 1 = yes, 0 = no |

**Row Count:** ~45,000 subscription records (monthly snapshots per account)

---

## 💰 **3. MRR Movements Table**

| Column | Description |
|--------|-------------|
| **movement_id** | Unique revenue change event ID (e.g., `MOV_00001`) |
| **account_id** | Links to accounts table |
| **movement_date** | Date when revenue changed |
| **movement_type** | **New** (signup), **Expansion** (upgrade), **Contraction** (downgrade), **Churn** (cancellation) |
| **mrr_before** | MRR amount before change (0 for New customers) |
| **mrr_after** | MRR amount after change (0 for Churn) |
| **mrr_change** | Dollar change: Positive for New/Expansion, Negative for Contraction/Churn |
| **plan_tier_before** | Plan before change (NULL for New customers) |
| **plan_tier_after** | Plan after change (NULL for Churn) |

**Row Count:** ~2,500 movements
- **New:** 2,000 (one per account signup)
- **Expansion:** ~200 (upgrades)
- **Contraction:** ~50 (downgrades)
- **Churn:** ~240 (cancellations)

---

## ⚠️ **4. Churn Events Table**

| Column | Description |
|--------|-------------|
| **churn_id** | Unique churn event ID (e.g., `CHN_00001`) |
| **account_id** | Links to accounts table |
| **subscription_id** | Final subscription before churn |
| **churn_date** | Date customer cancelled |
| **reason_code** | Cancellation reason: Price Too High, Missing Features, Switched to Competitor, No Longer Needed, Poor Customer Support |
| **mrr_lost** | Revenue lost from this cancellation |
| **preceding_upgrade_flag** | 1 if customer upgraded before churning, 0 otherwise |
| **preceding_downgrade_flag** | 1 if customer downgraded before churning, 0 otherwise |
| **is_reactivation** | 1 if this was a returning customer, 0 for first-time (all are 0 in this dataset) |
| **feedback_text** | Customer exit feedback (NULL ~15% - silent churners) |

**Row Count:** ~ 240 churn events (~12% churn rate)

---

## 🎫 **5. Support Tickets Table**

| Column | Description |
|--------|-------------|
| **ticket_id** | Unique ticket ID (e.g., `TKT_00001`) |
| **account_id** | Links to accounts table |
| **created_date** | When ticket was opened |
| **resolved_date** | When ticket was closed (NULL if unresolved) |
| **priority** | Urgency level: Low, Medium, High, Critical |
| **category** | Issue type: Bug, Feature Request, How-To, Billing, Performance, Integration |
| **resolution_time_hours** | Time to resolve in hours |
| **satisfaction_score** | Customer rating 1-5 (NULL ~20% - survey not filled) |
| **escalation_flag** | 1 if escalated to management, 0 otherwise |

**Row Count:** 17,000 tickets (avg 8-9 per account)
- Churned accounts have higher ticket volume (6-15/year)
- Active accounts have lower volume (2-8/year)

---

## 🖱️ **6. Feature Usage Table**

| Column | Description |
|--------|-------------|
| **usage_id** | Unique usage event ID (e.g., `USG_000001`) |
| **account_id** | Links to accounts table |
| **feature_name** | Feature used: Dashboard Export, API Access, Team Collaboration, Custom Reports, Integrations, Advanced Analytics, Mobile App, SSO, White Labeling, Webhook, Bulk Import, Auto Backup, Custom Fields, Role Permissions, Email Automation |
| **usage_date** | Date of usage |
| **usage_count** | Number of times feature was used that day |
| **usage_duration_secs** | Total time spent in feature (seconds) |
| **error_count** | Number of errors encountered (0 or 1 typically) |
| **is_beta_feature** | 1 if experimental feature, 0 if stable |

**Row Count:** ~2.2M usage events
- **Ghosting pattern built in:** Usage declines 60 days before churn
- Engagement levels: Low (25%), Medium (50%), High (25%)

---

## 🔗 **Table Relationships**

```
accounts (1) ─────< (many) subscriptions
accounts (1) ─────< (many) mrr_movements
accounts (1) ─────< (many) churn_events
accounts (1) ─────< (many) support_tickets
accounts (1) ─────< (many) feature_usage
```

**All tables link via `account_id`**

---

## 📈 **Dataset Summary Stats**

| Metric | Value |
|--------|-------|
| **Total Accounts** | 2,000 |
| **Active Accounts** | ~1,760 (88%) |
| **Churned Accounts** | ~240 (12%) |
| **Date Range** | 2022-01-01 to 2024-12-31 (3 years) |
| **Total MRR Movements** | ~2,500 events |
| **Avg Tickets/Account** | 8.5 |
| **Total Feature Usage Events** | 2.2M |

---

**✅ All data is realistic, based on SaaS industry benchmarks, and ready for analysis!** 🚀